# Sensitivity of SPB Targets to Data and Assumptions

This notebook tests the sensitivity of SPB targets to the data and the model assumptions in the DSA model. The baseline is close to the reference trajectories of the Commission prior guidance of 2024: it uses the input workbook built from the Commission calculation sheets and the Commission implementation of the rules (`rules='commission'`).

Each run records the SPB required by each criterion, and every effect is reported in three tiers:

- the **own criterion**: the SPB required by the criterion a check acts on (a stress test for its size, the stochastic criterion for the stochastic settings, otherwise the deterministic debt criterion of the baseline scenario);
- the **DSA target**: the SPB required by all DSA criteria (debt declines or stays below 60% in the deterministic scenarios, deficit below 3%, stochastic criterion);
- the **binding target**: the SPB after the excessive deficit procedure (EDP), the debt sustainability safeguard and the deficit resilience safeguard are added.

The settings fall into blocks by what settles them (`BLOCKS` in `code/functions/sensitivity.py`):

1. **Forecasts that set the centre line** (`FORECAST_ASSUMPTIONS`): real short- and long-term rates and inflation at T+10, potential growth and ageing costs. Settled by evidence: the ranges are the interquartile range of historical forecast errors and of the Ageing Report scenarios (section 0). The fan chart of the stochastic analysis is centred on this line, and the deterministic criteria, which set most targets, take it as given.
2. **Model parameters** (`PARAMETER_ASSUMPTIONS`): fiscal multiplier, its persistence and the budget semi-elasticity. Settled by the literature (`LITERATURE_RANGES`).
3. **Estimation of risk**: start and horizon of the stochastic projection, shock sample, frequency, distribution and winsorisation. Settled by statistical practice.
4. **Risk standard**: probability threshold, size of the stress tests and ageing cost period. They have no true value and are shown as a menu with the price of each choice (`build_menu_specs`).

**Data and definitions**: the data update, stock-flow adjustments after T (non-zero after the forecast years only for Finland, Luxembourg and Greece, a question of gross versus net debt) and the repayment profile of debt.

## Main findings

To be updated with the results of the redesigned analysis.

## TOC
0. **Calibration of the ranges** of the forecasts (historical forecast errors, Ageing Report scenarios) and the model parameters (literature).
1. **Individual checks.** Each check changes one setting at a time, grouped by block; the risk standard as a menu with prices. Runs with nine other random seeds give a noise band.
2. **Global sensitivity analysis.** Joint draws of blocks 1 and 2, all other settings at the baseline: band of targets by country in three tiers, debt paths and an optional variance decomposition by block.
3. **Results tables**, displayed and saved to `output/sensitivity/`.
4. **Annex.** Individual checks for a 7-year adjustment period, and the individual checks and menu with the September 2026 data.

Runs are cached in `output/sensitivity/`: with `RUN = True`, the notebook computes only runs that are not in the cache. The individual checks take about 50 minutes on 12 cores, the 500 global draws about 3.5 hours (all runs include the stochastic criterion, about 10 seconds per run). `QUICK = True` runs a subset of countries and fewer global draws for testing. The helper functions are in `code/functions/sensitivity.py`.

For comments and suggestions please contact lennard.welslau[at]gmail[dot]com.\
Last update: October 2026

In [ ]:
# Import libraries and modules
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown
plt.rcParams.update({'axes.grid': True, 'grid.color': 'black', 'grid.alpha': 0.15, 'grid.linestyle': '--', 'font.size': 12,
                     'figure.dpi': 100, 'axes.titleweight': 'bold'})

# Import the DSA model classes and functions from the code folder
sys.path.append(os.path.abspath('..'))
from classes import StochasticDsaModel as DSA
from data_pipeline import EU27, latest_input_file
from functions.sensitivity import *

# Set autoreload
%load_ext autoreload
%autoreload 2

## Settings

In [ ]:
INPUT_FILE = 'dsa_inputs_commission_2024.xlsx'  # baseline input workbook (Commission prior guidance 2024)
BASELINE_RULES = 'commission'                    # rules of find_spb_binding in the baseline
MULTIPLIER_TYPE = 'pers'                         # fiscal multiplier with persistent effect in all runs (None: input file)
ADJUSTMENT_PERIODS = (4, 7)                      # adjustment periods of the individual checks
MAIN_PERIOD = 4                                  # period of the global analysis, text, charts and tables (others: annex)
COUNTRIES = EU27
SEED = 0                                         # random seed set before every run (noise band: seeds 1 to 9)
N_DRAWS = 500                                    # draws of the global sensitivity analysis (JOINT_RUNS)
REAL_RATES = True                                # rate assumptions are real rates: nominal rates move with inflation
ADJUSTMENT_BOUND = 5                             # largest annual adjustment searched (pp.); safeguard infeasible above
MAX_WORKERS = None                               # worker processes, None uses all CPUs

QUICK = False                                    # True: subset of countries and fewer global draws (for testing)
QUICK_COUNTRIES = ['BEL', 'DEU', 'ESP', 'FRA', 'ITA', 'NLD']
RUN = True                                       # True: compute runs missing from the cache; False: use cached runs only

if QUICK:
    COUNTRIES, N_DRAWS = QUICK_COUNTRIES, 40

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
run_options = dict(countries=COUNTRIES, adjustment_periods=ADJUSTMENT_PERIODS, input_file=INPUT_FILE,
                   rules=BASELINE_RULES, seed=SEED, run=RUN, max_workers=MAX_WORKERS,
                   model_kwargs={'fiscal_multiplier_type': MULTIPLIER_TYPE} if MULTIPLIER_TYPE else None,
                   attributes={'adjustment_bound': ADJUSTMENT_BOUND} if ADJUSTMENT_BOUND != 3 else None)

## 0. Calibration of the ranges

The forecasts that set the centre line of the projection are varied over the interquartile range of historical forecast errors (realised minus forecast), the same range for all countries. Rates, inflation and potential growth are shifted at the T+10 anchor and converge back to the baseline by T+30, as in the model's interpolation between anchors.

- **Interest rates:** German zero-coupon curve (Bundesbank). The DSA uses market forward rates for T+10; the errors are the 1-year and 10-year rates realised ten years later minus the forward rates priced at each year end (1986-2015). Only Germany has a public yield curve long enough; the windows overlap. The errors are nominal but shift the real rates, and inflation is shifted on top (nominal rates move with inflation), so the range of nominal rates in the checks is somewhat wider than in the data. Forward rates were on average well above the realised rates (term premium and the long fall in rates); this bias is shown but not used.
- **Inflation and growth:** IMF WEO forecasts (Mendeley Data panel of all vintages), average error over the five forecast years of each autumn vintage 2000-2018, 27 EU countries. The errors are demeaned by country; the average error (a bias: growth was over-forecast) is shown but not used. The growth errors are errors in actual growth, so the range is generous for a shift in potential growth.
- **Ageing costs:** deviation of the 2024 Ageing Report scenarios (all except the pension policy scenarios: life expectancy, migration, fertility, employment of older workers, productivity growth and the risk scenario for health care and long-term care) from the baseline change in ageing costs between 2024 and 2038 (the end of the DSA horizon for a 4-year adjustment), all countries and scenarios. The deviation grows linearly from T (`shift_ageing_cost`).

The checks use the rounded values in `CALIBRATED_RANGES`, so that the run caches do not depend on data revisions; the table compares them with the ranges recomputed from the data. The WEO data are downloaded to `data/RawData/reference/` if missing; the forward rates are read from the Bundesbank API.

In the global analysis (section 2), the short- and long-term rates are drawn as one factor (`COMPOSITE_ASSUMPTIONS`): both are set at the same position in their ranges, since their errors come from the same yield curve and are highly correlated. The other assumptions are drawn independently.

The ranges of the model parameters come from the literature (`LITERATURE_RANGES`): fiscal multiplier 0.5 to 1.5 (most spending multipliers lie between 0.6 and 1; consolidations in Europe after 2010 and multipliers without a monetary policy response, as for one country in a monetary union, about 1.5), persistence of its effect 2 to 5 years (two-year cumulative multipliers at least as large as impact multipliers; lasting effects of consolidations) and the budget semi-elasticity scaled by 0.9 to 1.1 (the largest revisions of the Commission estimates). Sources are listed in the code.

In [ ]:
try:
    calibration_ranges = calibration_table()
    calibration_ranges.to_csv(OUTPUT_DIR / 'calibration_ranges.csv')
except Exception as err:  # no access to the WEO, Bundesbank or Commission data: use the saved table
    print(f'Data not available ({type(err).__name__}); using the saved calibration table')
    calibration_ranges = pd.read_csv(OUTPUT_DIR / 'calibration_ranges.csv', index_col=0)
display(calibration_ranges.round(2))
gap = (calibration_ranges[['p25', 'p75']].values - calibration_ranges[['range_low', 'range_high']].values)
print('Ranges used match the data:', bool(abs(gap).max() < 0.015))

## 1. Individual checks

### 1.1 Specification table

Each row changes one setting relative to the baseline. Checks are grouped by block: forecasts, model parameters, estimation of risk, risk standard (with the lenient and strict scenarios, in which all its settings are at one end of their range) and data and definitions. Input changes are relative to the baseline workbook: the notebook reads the baseline value with `read_country` and passes the new absolute value as an override. Interest rates, inflation and potential growth are shifted at the T+10 anchor over the calibrated ranges (section 0) and converge back to the baseline by T+30, with the linear interpolation the model uses between anchors (`shift_anchor`, `shift_potential_growth_t10`); potential growth moves together with baseline real growth, so that the output gap is unchanged. Separate checks shift the T+30 anchors, for which there is no evidence on errors. The ageing-cost check adds the calibrated deviation to the change in ageing costs (`shift_ageing_cost`); the stock-flow check sets the stock-flow adjustment to zero after T. With `REAL_RATES = True`, interest rate checks shift real rates: an inflation check moves the nominal short- and long-term rates by the same amount, so that real rates are unchanged.

In [ ]:
oat_specs = build_oat_specs(latest_file=latest_input_file(), real_rates=REAL_RATES)
with pd.option_context('display.max_rows', 200, 'display.max_colwidth', 70):
    display(spec_table(oat_specs))

### 1.2 Runs

The baseline and every check run for each country and adjustment period. The Commission targets are added for comparison: the published SPB at the end of the adjustment period (after the safeguards) and the target from the DSA criteria alone.

In [ ]:
oat = run_specs(oat_specs, cache_file=OUTPUT_DIR / 'oat_runs.pkl', **run_options)
vintage = 'latest' if 'latest' in INPUT_FILE else '2024'
oat = oat.merge(commission_targets(COUNTRIES, vintage=vintage), on=['country', 'adjustment_period'], how='left')
oat = add_own_criterion(add_deviations(oat))
noise = noise_band(oat, target='dsa')
checks = oat[oat['group'] != 'Baseline']
print(f"{len(oat)} runs, {oat['error'].notna().sum() if 'error' in oat else 0} failed")

### 1.3 Baseline and Commission targets

The baseline is compared with the Commission targets, both from the DSA criteria alone and after the safeguards. The model reproduces them closely with the Commission's output gap closure rule (`MULTIPLIER_TYPE = None`, see `commission_replication.ipynb`); the persistent multiplier used here moves some targets by a few tenths of a percentage point.

In [ ]:
base = oat[(oat['id'] == 'baseline') & (oat['adjustment_period'] == MAIN_PERIOD)].copy()
base['dsa difference'] = base['dsa_target'] - base['commission_dsa_target']
base['binding difference'] = base['binding_target'] - base['commission_target']
table = base.set_index('country')[['commission_dsa_target', 'dsa_target', 'dsa difference', 'commission_target',
                                   'binding_target', 'binding difference', 'binding_criterion']]
display(table.round(2))
print(f"Median absolute difference {base['dsa difference'].abs().median():.2f} pp. (DSA target), "
      f"{base['binding difference'].abs().median():.2f} pp. (binding target); "
      f"maximum {base['dsa difference'].abs().max():.2f} and {base['binding difference'].abs().max():.2f} pp.")

### 1.4 Main table: effects of each check

Change of the SPB target relative to the baseline for each check variant, in pp. of GDP: mean, minimum and maximum across countries and the number of countries whose target changes by more than 0.1 pp. (whether an effect is broad or driven by a few countries), for the DSA target (DSA criteria only) and the binding target (DSA criteria, EDP and safeguards). The noise band from alternative seeds is printed above the table; effects within it are indistinguishable from simulation noise. Countries without data for a check (no Eurostat repayment profile for Hungary, Luxembourg and the Netherlands) are left out of its statistics.

In [ ]:
main_table = effects_table(oat, (MAIN_PERIOD,))
main_noise = noise[noise['adjustment_period'] == MAIN_PERIOD]
print(f"Noise band (median half-width across countries, DSA target): {float((main_noise['noise_high'] - main_noise['noise_low']).median() / 2):.3f} pp.")
with pd.option_context('display.max_rows', 200):
    display(main_table.round(2))

### 1.5 Which checks matter most

Mean change of the SPB target across countries for each check variant: bars for the DSA target, diamonds for the binding target. Means keep the sign of the changes, so a check that raises the target in some countries and lowers it in others can have a small mean; the range across countries is in the main table.

In [ ]:
plot_mean_effects(oat, (MAIN_PERIOD,))
plt.show()

### 1.6 Risk standard: menu with prices

The probability threshold, the size of the stress tests and the ageing cost period have no true value: they set how much risk the rules accept. They are shown as a menu, with the price of each choice: the change of the SPB target relative to the baseline (mean across countries, number of countries whose target changes by more than 0.1 pp., and the largest change). The stochastic criterion is shown as a grid of threshold, start and horizon of the stochastic projection. The start and the horizon are modelling choices (section 1.4), but they set how much the threshold costs: with the baseline start and horizon, the stochastic criterion binds for few countries, so the threshold is cheap. Runs with the same settings as an individual check are read from the cache (`build_menu_specs`, `menu_table`).

In [ ]:
menu = add_deviations(run_specs(build_menu_specs(), cache_file=OUTPUT_DIR / 'oat_runs.pkl', **run_options))
menu_prices = menu_table(menu, MAIN_PERIOD)
display(menu_prices.round(2))
# Price by country: change of the DSA target for each choice
menu_by_country = (menu[(menu['adjustment_period'] == MAIN_PERIOD) & (menu['group'] == RISK_STANDARD)]
                   .pivot_table(index='country', columns=['check', 'variant'], values='dsa_delta_target', sort=False))
with pd.option_context('display.max_columns', 50):
    display(menu_by_country.round(2))

## 2. Global sensitivity analysis

Individual checks miss interactions: a higher interest rate may matter more once growth is low, and a large multiplier matters more when its effect is persistent. The global analysis draws the forecasts (block 1) and the model parameters (block 2) at once (`JOINT_RUNS`); the estimation of risk, the risk standard and the data stay at the baseline. Each input is drawn uniformly over its range in the individual checks (its interquartile range of errors for the forecasts, its literature range for the parameters), the persistence of the multiplier with equal probability for each value. The draws are independent, except that the short- and long-term rates are one factor (`COMPOSITE_ASSUMPTIONS`); interest rate draws are real rates (`REAL_RATES`). Each draw is a full run with the stochastic criterion, for the 4-year adjustment period. The annual adjustment is searched up to `ADJUSTMENT_BOUND` (5 pp.); draws in which the debt sustainability safeguard cannot be met even at this bound are flagged as infeasible (`safeguard_infeasible`) and left out of the percentiles of the binding target.

Since each input varies within its interquartile range, the band of targets shows the uncertainty when each forecast misses by no more than a typical error: a likely range, not a worst case. For each country, the 10th and the 90th percentile of the draws give an accommodative and a restrictive combination.

In [ ]:
joint = {}
for name, assumptions in JOINT_RUNS.items():
    specs = build_global_specs(n_draws=N_DRAWS, seed=SEED, assumptions=assumptions, real_rates=REAL_RATES)
    runs = run_specs(specs, cache_file=OUTPUT_DIR / f'global_runs_{name}.pkl', **{**run_options, 'adjustment_periods': (MAIN_PERIOD,)})
    joint[name] = add_own_criterion(add_deviations(runs))
glob = joint['joint']
draws = glob[glob['group'] == 'Global']
combinations = draw_combinations(glob)
results = add_own_criterion(add_deviations(pd.concat([oat, combinations], ignore_index=True)))
combined_table = effects_table(combinations, (MAIN_PERIOD,))
display(combined_table.round(2))

### 2.1 Range of targets relative to the required adjustment

By country: the adjustment that the baseline binding target requires (binding target minus the SPB in T); for each tier (own criterion, DSA target, binding target) the band between the 10th and 90th percentile of the draws and the change of the 10th and 90th percentile relative to the baseline; the position of the baseline in the draws (share of draws with a lower target); the number of infeasible draws and the share of draws whose binding target requires a total adjustment of more than 6 pp.

In [ ]:
q_lo, q_hi = COMBINATIONS['draws:accommodative'][1], COMBINATIONS['draws:restrictive'][1]
base_main = oat[(oat['id'] == 'baseline') & (oat['adjustment_period'] == MAIN_PERIOD)].set_index('country')
rows = []
for c in COUNTRIES:
    dc = draws[draws['country'] == c]
    row = {'country': c, 'required adjustment': base_main.loc[c, 'binding_target'] - base_main.loc[c, 'spb_T']}
    for tier, (label, col) in TIERS.items():
        x = feasible(dc, tier)[col].dropna()
        row[f'{tier} P10'], row[f'{tier} P90'] = x.quantile(q_lo), x.quantile(q_hi)
        row[f'{tier} band'] = row[f'{tier} P90'] - row[f'{tier} P10']
        if tier != 'own':
            row[f'{tier} baseline rank'] = (x < -1e-9).mean()
    row['infeasible draws'] = int(dc['safeguard_infeasible'].sum())
    row['share > 6 pp.'] = ((dc['binding_target'] - dc['spb_T']) > 6 + 1e-9).mean()
    rows.append(row)
bands = pd.DataFrame(rows).set_index('country')
display(bands.round(2))
adjust = bands['required adjustment'] > 0
print('Median, countries that must adjust:', bands[adjust].median().round(2).to_dict())

### 2.2 Which block matters (optional)

The variance of the target across draws is split by country with squared standardised regression coefficients, summed by block (forecasts, model parameters): the share of the variance explained linearly by each block. The squared coefficients are rescaled to add up to the R² of the regression; the rest is due to non-linear effects and interactions, mostly switches of the binding criterion and the debt sustainability safeguard. For the binding target, infeasible draws are left out. The table aggregates over countries, weighting each country by the variance of its target; the shares by country are saved in `output/sensitivity/sensitivity_results.xlsx`.

In [ ]:
vd = pd.concat([variance_decomposition(draws, target=t, assumptions=JOINT_ASSUMPTIONS, categories=GLOBAL_CATEGORIES)
                for t in ['dsa', 'binding']], ignore_index=True)
variance = variance_table(vd, JOINT_ASSUMPTIONS, GLOBAL_CATEGORIES)
display(variance.round(2))

### 2.3 Range of targets by country

Targets by country, ordered by the baseline target. The grey bar spans the targets of the individual checks of the forecasts and the model parameters; green and red markers show the 10th and the 90th percentile of the draws (binding target without infeasible draws). Markers also show the baseline, the Commission target and the SPB in T; the distance between the SPB in T and a target is the adjustment the target requires. DSA targets first, then binding targets (Commission target as published).

In [ ]:
for target in ['dsa', 'binding']:
    plot_ranges(results[~results['group'].isin([ESTIMATION, RISK_STANDARD, DATA])], MAIN_PERIOD, target=target)
    plt.show()

### 2.4 Debt paths by country

Debt under a linear adjustment to the DSA target of the baseline and of the draws closest to the 10th and the 90th percentile of the draws (green, red), with the range across the individual checks of the forecasts and the model parameters in grey.

In [ ]:
plot_debt_paths(results[~results['group'].isin([ESTIMATION, RISK_STANDARD, DATA])], adjustment_period=MAIN_PERIOD, target='dsa')
plt.show()

### 2.5 Risk standard scenarios

Targets by country with the baseline risk standard, the probability target at 60% and 80%, and the lenient and strict scenarios (`CALIBRATION_SCENARIOS`), in which all settings of the risk standard are at one end of their range.

In [ ]:
cal_ids = ['baseline', 'stochastic:prob_target:0.6', 'stochastic:prob_target:0.8', 'calibration:lenient', 'calibration:strict']
cal = oat[(oat['adjustment_period'] == MAIN_PERIOD) & oat['id'].isin(cal_ids)]
calibration = pd.concat({t: cal.pivot(index='country', columns='id', values=TARGETS[t]['target'])[cal_ids]
                         for t in ['dsa', 'binding']}, axis=1)
display(calibration.round(2))
print('Mean change relative to the baseline:', (calibration.sub(calibration.xs('baseline', axis=1, level=1), level=0)).mean().round(2).to_dict())

## 3. Results tables

The main result tables, also saved to `output/sensitivity/sensitivity_results.xlsx` together with the tidy results (one row per run, sheets `runs`, `global_runs_economy` and `global_runs_economy_measurement`) and the variance shares by country. Columns with dictionaries (SPB targets by criterion, debt paths) are kept in the cached runs (`*.pkl`).

In [ ]:
# Result tables
tables = {
    'Calibration of the ranges (0)': calibration_ranges,
    'Baseline and Commission targets (1.3)': table,
    'Effects of each individual check (1.4)': main_table,
    'Risk standard: price of each choice (1.6)': menu_prices,
    '10th and 90th percentile of the global draws (2)': combined_table,
    'Range of targets relative to the required adjustment (2.1)': bands,
    'Variance decomposition by block (2.2)': variance,
    'Risk standard scenarios (2.5)': calibration,
    'Specifications of the individual checks': spec_table(oat_specs),
}
for title, t in tables.items():
    display(Markdown(f'**{title}**'))
    with pd.option_context('display.max_rows', 200, 'display.max_colwidth', 70):
        display(t.round(2))

# Save, with the annex table (section 4)
annex = effects_table(oat, tuple(n for n in ADJUSTMENT_PERIODS if n != MAIN_PERIOD))
flat = lambda df: df.drop(columns=[c for c in ['spb_target_dict', 'debt_path', 'dsa_debt_path'] if c in df])
sheets = {'calibration_ranges': calibration_ranges, 'menu': menu_prices, 'menu_by_country': menu_by_country, 'effects': main_table, 'annex_effects': annex, 'global_combinations': combined_table, 'bands': bands,
          'global_variance': variance, 'calibration': calibration, 'baseline_commission': table,
          'specs': tables['Specifications of the individual checks']}
with pd.ExcelWriter(OUTPUT_DIR / 'sensitivity_results.xlsx') as writer:
    flat(results).to_excel(writer, sheet_name='runs', index=False)
    for name, t in sheets.items():
        t.to_excel(writer, sheet_name=name, index=not isinstance(t.index, pd.RangeIndex))
    for name, g in joint.items():
        flat(g).to_excel(writer, sheet_name=f'global_runs_{name}'[:31], index=False)
    vd.to_excel(writer, sheet_name='global_variance_by_country', index=False)
print(f"Saved to {OUTPUT_DIR / 'sensitivity_results.xlsx'}")

## 4. Annex: 7-year adjustment

Effects of each individual check on the SPB target for a 7-year adjustment period, as in the main table (section 1.4): mean, minimum and maximum change relative to the baseline across countries and the number of countries with a change above 0.1 pp., for the DSA target and the binding target (pp. of GDP). The global analysis covers the 4-year adjustment period only.

In [ ]:
with pd.option_context('display.max_rows', 200):
    display(annex.round(2))

## 5. Annex: September 2026 data

The individual checks and the menu of the risk standard rerun with the September 2026 data (reference year 2026, 4-year adjustment). The table compares the mean change of the targets across countries in the three tiers with the 2024 data, to show whether the effects depend on the conditions of 2024. Runs are cached in `output/sensitivity/oat_runs_latest.pkl`.

In [ ]:
latest = latest_input_file()
latest_specs = [s for s in oat_specs if s['id'] != 'data:latest' and not s['id'].startswith('noise:')]
latest_specs += [s for s in build_menu_specs() if s['id'] not in {x['id'] for x in latest_specs}]
oat_latest = add_own_criterion(add_deviations(run_specs(
    latest_specs, cache_file=OUTPUT_DIR / 'oat_runs_latest.pkl',
    **{**run_options, 'input_file': latest, 'adjustment_periods': (MAIN_PERIOD,)})))
both_2024 = add_own_criterion(add_deviations(pd.concat([oat, menu[menu['group'] == RISK_STANDARD]], ignore_index=True)))
vintages = pd.concat({'2024 data': effects_table(both_2024, (MAIN_PERIOD,)),
                      '2026 data': effects_table(oat_latest, (MAIN_PERIOD,))}, axis=1)
vintage_table = vintages.loc[:, [c for c in vintages.columns if c[-1] in ('mean', 'countries > 0.1 pp.')]]
vintage_table.columns = [f'{v} {t} {s}'.replace('countries > 0.1 pp.', 'N') for v, _, t, s in vintage_table.columns]
with pd.option_context('display.max_rows', 200, 'display.max_columns', 30):
    display(vintage_table.round(2))